# Llama Surgery

## Pre-trained experts -> Final Layers - > MoE

We take a donor model, and trained experts. We extract the final layers from the experts, turn these into experts, and replace the model's final layers.

In [ ]:
# Install Once
!pip install huggingface_hub[hf_xet]
!pip install -U bitsandbytes
!pip install -U peft

In [ ]:
## 📦 Environment Setup: Dependencies and Imports

import torch
import time
import os
import subprocess
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig
import gc
import sys
import importlib
from torch import nn
from torch.nn import functional as F
from copy import deepcopy
from peft import PeftModel, PeftConfig, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training


In [ ]:
# Required packages
required_packages = [
    'torch', 'transformers', 'datasets', 'accelerate', 'flash_attn',
    'evaluate', 'lm_eval', 'sklearn', 'matplotlib', 'wandb',
    'tqdm', 'sentencepiece', 'scipy', 'einops'
]

# Check and install missing packages
for package in required_packages:
    try:
        module = importlib.import_module(package)
        print(f"✅ {package} installed successfully")
        if package == 'torch':
            print(f"   Version: {torch.__version__}")
            print(f"   CUDA available: {torch.cuda.is_available()}")
            if torch.cuda.is_available():
                print(f"   CUDA version: {torch.version.cuda}")
                print(f"   GPU: {torch.cuda.get_device_name(0)}")
        elif hasattr(module, '__version__'):
            print(f"   Version: {module.__version__}")
    except ImportError:
        print(f"❌ {package} not found. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        module = importlib.import_module(package)
        print(f"✅ {package} installed successfully (post-install)")
        if hasattr(module, '__version__'):
            print(f"   Version: {module.__version__}")

# You may need to restart the Kernel to use these

✅ torch installed successfully
   Version: 2.6.0+cu124
   CUDA available: True
   CUDA version: 12.4
   GPU: NVIDIA A100-SXM4-40GB
✅ transformers installed successfully
   Version: 4.51.3
✅ datasets installed successfully
   Version: 3.6.0
✅ accelerate installed successfully
   Version: 1.6.0
✅ flash_attn installed successfully
   Version: 2.7.4.post1
✅ evaluate installed successfully
   Version: 0.4.3
✅ lm_eval installed successfully
✅ sklearn installed successfully
   Version: 1.6.1
✅ matplotlib installed successfully
   Version: 3.10.0
✅ wandb installed successfully
   Version: 0.19.10
✅ tqdm installed successfully
   Version: 4.67.1
✅ sentencepiece installed successfully
   Version: 0.2.0
✅ scipy installed successfully
   Version: 1.15.2
✅ einops installed successfully
   Version: 0.8.1


# 🧠 Model Architecture: Llama-3-8B-UltraMedical

**Llama-3-8B-UltraMedical** is built on top of the LLaMA 3 architecture and fine-tuned on the UltraMedical dataset. In this notebook, we'll walk through its architecture and prepare it for Mixture-of-Experts (MoE) insertion.


## 🔍 Step 1: Load and Inspect Model Layers

We begin by loading the model to examine its transformer block structure.


In [ ]:
# Shared Tokenizer Path
shared_tokenizer_path = "/content/drive/MyDrive/medmoe/shared_tokenizer"
adapter_path = "/content/drive/MyDrive/medmoe/checkpoints/cardiology_bf16_llama3_8b_expert_model/checkpoint-160"

# Set Expert Paths
base_model_path = "TsinghuaC3I/Llama-3-8B-UltraMedical"
expert_3 = "/content/drive/MyDrive/medmoe/checkpoints/cardiology_bf16_llama3_8b_expert_model/checkpoint-160"
expert_2 = "/content/drive/MyDrive/medmoe/checkpoints/mental_health_bf16_llama3_8b_expert_model/checkpoint-80"

expert_1_model = "/content/drive/MyDrive/medmoe/model/cardiology_bf16_llama3_8b_expert_model"
expert_1_cp = "/content/drive/MyDrive/medmoe/checkpoints/cardiology_bf16_llama3_8b_expert_model/checkpoint-160"


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/peft/peft_model.py:569: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_B.default.we

# Load base Expert

We will load an expert to use as the donor

In [ ]:
# === Load base model ===
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    device_map="cuda",
    torch_dtype = torch.bfloat16,
    trust_remote_code=True
)

# === Load PEFT adapter ===
model = PeftModel.from_pretrained(
    base_model,
    expert_3)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['_manually_filtered'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/peft_model.py:569: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0

## Examine the transformer layers

In [ ]:
# Peek at the transformer layers
transformer_layers = model.model.layers
print(f"Number of layers: {len(transformer_layers)}")
for idx,layer in enumerate(transformer_layers):
    print(f"Layer Index {idx}:\n {layer}")

# 🧩 MoE_LlamaMLP: Replacing the FFN with a Mixture of Experts

In this section, we define `MoE_LlamaMLP`, a drop-in replacement for the original `LlamaMLP`. It supports:

- 3 parallel FFN experts
- A lightweight Router MLP
-

The original `mlp` block looks like this:

```
LlamaMLP(
  (gate_proj): Linear(4096 → 14336)
  (up_proj): Linear(4096 → 14336)
  (down_proj): Linear(14336 → 4096)
  (act_fn): SiLU()
)
```

In [ ]:
# Expert wrapper
class Expert(nn.Module):
    def __init__(self, layers, name):
        super().__init__()
        self.name = name
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        for layer in self.model:
            x = layer.mlp(x)  # Only the MLPs
        return x

## Choose final layers to use as experts

In [ ]:
# Choose what layer to take experts 16-31
start_expert_layers = 28

# Create Expert 1 with deepcopy (so it's independent)
copy_layers = list(model.model.layers[start_expert_layers:])

expert1 = Expert(deepcopy(copy_layers), f"Expert1_Orthopedic_{start_expert_layers}+")

In [ ]:
# Load Expert 2

# Load Expert Donor
model2 = AutoModelForCausalLM.from_pretrained(expert_2,
                                             device_map="auto",
                                             torch_dtype=torch.float16,
                                             trust_remote_code=True)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['_manually_filtered'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(
Loading adapter weights from /content/drive/MyDrive/medmoe/model/mental_health_bf16_llama3_8b_expert_model led to unexpected keys not found in the model: model.layers.0.self_attn.q_proj.lora_A.default.default.weight, model.layers.0.self_attn.q_proj.lora_B.default.default.weight, model.layers.0.self_attn.k_proj.lora_A.default.default.weight, model.layers.0.self_attn.k_proj.lora_B.default.default.weight, model.layers.0.self_attn.v_proj.lora_A.default.default.weight, model.layers.0.self_attn.v_proj.lora_B.default.default.weight, model

In [ ]:
# Extract Layers from expert 2

# Create Expert  with deepcopy (so it's independent)
copy_layers = list(model2.model.layers[start_expert_layers:])

expert2 = Expert(deepcopy(copy_layers), "Expert2_MentalHealth_28+")

In [ ]:
# Offload model2 for GPU memory
del model2
del copy_layers
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Load Expert 3

# Load Expert Donor
model3 = AutoModelForCausalLM.from_pretrained(expert_3,
                                             device_map="auto",
                                             torch_dtype=torch.float16,
                                             trust_remote_code=True)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['_manually_filtered'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(
Loading adapter weights from /content/drive/MyDrive/medmoe/model/cardiology_bf16_llama3_8b_expert_model led to unexpected keys not found in the model: model.layers.0.self_attn.q_proj.lora_A.default.default.weight, model.layers.0.self_attn.q_proj.lora_B.default.default.weight, model.layers.0.self_attn.k_proj.lora_A.default.default.weight, model.layers.0.self_attn.k_proj.lora_B.default.default.weight, model.layers.0.self_attn.v_proj.lora_A.default.default.weight, model.layers.0.self_attn.v_proj.lora_B.default.default.weight, model.la

In [ ]:
# Extract Layers from expert 3

# Create Expert  with deepcopy (so it's independent)
copy_layers = list(model3.model.layers[start_expert_layers:])

expert3 = Expert(deepcopy(copy_layers), "Expert3_Cardiology_28+")

In [ ]:
# Offload model3
del model3
del copy_layers
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Combine experts
experts = [expert1,expert2,expert3]

In [ ]:
# Memory cleanup
gc.collect()
torch.cuda.empty_cache()
time.sleep(5)

del copy_layers
gc.collect()
torch.cuda.empty_cache()

In [ ]:
gc.collect()
torch.cuda.empty_cache()

# Based on DeepMoE class below, choose softmax routing or uniform all shared experts.

In [ ]:
# DeepMoE with soft routing
class DeepMoE(nn.Module):
    def __init__(self, hidden_size, experts):
        super().__init__()
        self.num_experts = len(experts)

        self.gate = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, self.num_experts)
        )
        self.experts = nn.ModuleList(experts)

    def forward(self, x):
        gate_logits = self.gate(x)                     # [B, T, E]
        weights = torch.softmax(gate_logits, dim=-1)   # [B, T, E]
        outputs = [expert(x) for expert in self.experts]  # [B, T, H]
        stacked = torch.stack(outputs, dim=-1)            # [B, T, H, E]
        return (weights.unsqueeze(2) * stacked).sum(-1)   # [B, T, H]

In [ ]:
# DeepMoE with UNIFORM routing
class DeepMoE(nn.Module):
    def __init__(self, hidden_size, experts):
        super().__init__()
        self.num_experts = len(experts)

        self.gate = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, self.num_experts)
        )
        self.experts = nn.ModuleList(experts)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape

        # 🔁 Uniform weights for all experts
        weights = torch.full(
            (batch_size, seq_len, self.num_experts),
            1.0,
            #1.0 / self.num_experts,
            device=x.device,
            dtype=x.dtype
        )  # [B, T, E]

        # 🔁 Each expert gets same input
        outputs = [expert(x) for expert in self.experts]  # [B, T, H] per expert

        stacked = torch.stack(outputs, dim=-1)            # [B, T, H, E]
        #return (weights.unsqueeze(2) * stacked).sum(-1)   # [B, T, H]
        return outputs[0]

In [ ]:
# Delete layers from model to prevent reuse
for i in range(start_expert_layers, len(model.model.layers)-1):
    del model.model.layers[i]

In [ ]:
# Verify Deletion
# 🔍 Inspect Transformer Layers After MoE Injection

transformer_layers = model.model.layers  # 32 decoder blocks

for i, layer in enumerate(transformer_layers):
    layer_name = getattr(layer, 'name', f"LlamaDecoderLayer_{i}")
    total_params = sum(p.numel() for p in layer.parameters())
    print(f"Layer {i}: {layer_name} with {total_params:,} parameters")

Layer 0: LlamaDecoderLayer_0 with 219,422,720 parameters
Layer 1: LlamaDecoderLayer_1 with 219,422,720 parameters
Layer 2: LlamaDecoderLayer_2 with 219,422,720 parameters
Layer 3: LlamaDecoderLayer_3 with 219,422,720 parameters
Layer 4: LlamaDecoderLayer_4 with 219,422,720 parameters
Layer 5: LlamaDecoderLayer_5 with 219,422,720 parameters
Layer 6: LlamaDecoderLayer_6 with 219,422,720 parameters
Layer 7: LlamaDecoderLayer_7 with 219,422,720 parameters
Layer 8: LlamaDecoderLayer_8 with 219,422,720 parameters
Layer 9: LlamaDecoderLayer_9 with 219,422,720 parameters
Layer 10: LlamaDecoderLayer_10 with 219,422,720 parameters
Layer 11: LlamaDecoderLayer_11 with 219,422,720 parameters
Layer 12: LlamaDecoderLayer_12 with 219,422,720 parameters
Layer 13: LlamaDecoderLayer_13 with 219,422,720 parameters
Layer 14: LlamaDecoderLayer_14 with 219,422,720 parameters
Layer 15: LlamaDecoderLayer_15 with 219,422,720 parameters
Layer 16: LlamaDecoderLayer_16 with 219,422,720 parameters
Layer 17: LlamaDe

In [ ]:
# Inject DeepMoE into model

# Instantiate DeepMoE
deepmoe = DeepMoE(
    hidden_size=4096,
    experts=experts
)

# Move DeepMoE to real device safely (handles meta tensors)
deepmoe = deepmoe.to_empty(device=model.device)

# Inject into model
model.model.layers[start_expert_layers].mlp = deepmoe

In [ ]:
# test and inspect
for i, expert in enumerate(experts):
    print(f"Expert {i}: {expert.name} with {sum(p.numel() for p in expert.parameters()):,} parameters")

Expert 0: Expert1_Orthopedic_28+ with 877,690,880 parameters
Expert 1: Expert2_MentalHealth_28+ with 877,690,880 parameters
Expert 2: Expert3_Cardiology_28+ with 877,690,880 parameters


In [ ]:
# Inspect new architecture
transformer_layers = model.model.layers
print(f"Number of layers: {len(transformer_layers)}")
for idx,layer in enumerate(transformer_layers):
    print(f"Layer {idx}:\n {layer}")

Number of layers: 29
Layer 0:
 LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): lora.Linear(
      (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
      (lora_dropout): ModuleDict(
        (default): Dropout(p=0.05, inplace=False)
      )
      (lora_A): ModuleDict(
        (default): Linear(in_features=4096, out_features=16, bias=False)
      )
      (lora_B): ModuleDict(
        (default): Linear(in_features=16, out_features=4096, bias=False)
      )
      (lora_embedding_A): ParameterDict()
      (lora_embedding_B): ParameterDict()
      (lora_magnitude_vector): ModuleDict()
    )
    (k_proj): lora.Linear(
      (base_layer): Linear(in_features=4096, out_features=1024, bias=False)
      (lora_dropout): ModuleDict(
        (default): Dropout(p=0.05, inplace=False)
      )
      (lora_A): ModuleDict(
        (default): Linear(in_features=4096, out_features=16, bias=False)
      )
      (lora_B): ModuleDict(
        (default): Linear(in_features=16

In [ ]:
# 🔍 Inspect Transformer Layers After MoE Injection

transformer_layers = model.model.layers  # 32 decoder blocks

for i, layer in enumerate(transformer_layers):
    layer_name = getattr(layer, 'name', f"LlamaDecoderLayer_{i}")
    total_params = sum(p.numel() for p in layer.parameters())
    print(f"Layer {i}: {layer_name} with {total_params:,} parameters")

Layer 0: LlamaDecoderLayer_0 with 219,422,720 parameters
Layer 1: LlamaDecoderLayer_1 with 219,422,720 parameters
Layer 2: LlamaDecoderLayer_2 with 219,422,720 parameters
Layer 3: LlamaDecoderLayer_3 with 219,422,720 parameters
Layer 4: LlamaDecoderLayer_4 with 219,422,720 parameters
Layer 5: LlamaDecoderLayer_5 with 219,422,720 parameters
Layer 6: LlamaDecoderLayer_6 with 219,422,720 parameters
Layer 7: LlamaDecoderLayer_7 with 219,422,720 parameters
Layer 8: LlamaDecoderLayer_8 with 219,422,720 parameters
Layer 9: LlamaDecoderLayer_9 with 219,422,720 parameters
Layer 10: LlamaDecoderLayer_10 with 219,422,720 parameters
Layer 11: LlamaDecoderLayer_11 with 219,422,720 parameters
Layer 12: LlamaDecoderLayer_12 with 219,422,720 parameters
Layer 13: LlamaDecoderLayer_13 with 219,422,720 parameters
Layer 14: LlamaDecoderLayer_14 with 219,422,720 parameters
Layer 15: LlamaDecoderLayer_15 with 219,422,720 parameters
Layer 16: LlamaDecoderLayer_16 with 219,422,720 parameters
Layer 17: LlamaDe

In [ ]:
from transformers import AutoTokenizer

#Load Shared Tokenizer
shared_tokenizer = AutoTokenizer.from_pretrained(shared_tokenizer_path,
                                          torch_dtype=torch.float16
                                          )

In [ ]:
# Input prompt
prompt = "What is hypertension?"
inputs = shared_tokenizer(prompt, return_tensors="pt").to(model.device)

# Forward pass through model
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    print("✅ Logits shape:", logits.shape)


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
output = model.generate(**inputs)
print(tokenizer.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


What is hypertension? Hyp Hyp Hyp Hyp Hyp Hyp Hyp Hyp Hyp Hyp Hyp Hyp Hyp Hyp Hyp Hyp Hyp Hyp Hyp Hyp


In [ ]:
print(output)

tensor([[ 3923,   374, 63308,    30, 39515, 39515, 39515, 39515, 39515, 39515,
         39515, 39515, 39515, 39515, 39515, 39515, 39515, 39515, 39515, 39515,
         39515, 39515, 39515, 39515]], device='cuda:0')


In [ ]:
# Log avg gate weights per expert
x = model.model.embed_tokens(inputs["input_ids"])
mlp = model.model.layers[16].mlp
with torch.no_grad():
    logits = mlp.gate(x)
    weights = torch.softmax(logits, dim=-1)
    print("🔍 Avg gate weights:", weights.mean(dim=[0, 1]))

AttributeError: 'LlamaMLP' object has no attribute 'gate'

In [ ]:
# Save our new MoE
from transformers import AutoTokenizer
import torch
import json
import os

# new_name
new_name = Llama3-UltraMedical-MoE-4x2.2B-9B
# ✅ Paths
save_path = "/content/drive/MyDrive/medmoe/MoE/"

# ✅ Save model weights (optionally use safetensors=True if needed)
ortho.save_pretrained(save_path)

# ✅ Save tokenizer
tokenizer.save_pretrained(save_path)

# ✅ Update config.json with correct metadata
# Update config
config_path = os.path.join(save_path, "config.json")

with open(config_path, "r") as f:
    config = json.load(f)

# Manual config patch
config.update({
    "architectures": ["LlamaForCausalLM"],
    "model_type": "llama",
    "torch_dtype": "float16",
    "use_cache": False,
    "moe_injected": True,
    "model_name": "Llama3-UltraMedical-MoE-4x2.2B-9B"
})

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

In [ ]:
# clear cache
gc.collect()
torch.cuda.empty_cache()